# Unit & data testing

Similarly to dbt, SQLMesh has unit tests and data tests. They are called slightly
differently:

- **Unit tests** are called **tests**
- **Data tests** are called **audits**

## Unit tests

Like dbt, SQLMesh has unit tests where you define certain (sample) inputs and expected
outputs, and SQLMesh checks that the model gives the expected output.

However, unlike dbt, SQLMesh parses your SQL and does 2 things with that:

1. It allows defining unit tests which don't just test the expected output of your
  entire query, but you can also verify the **expected output of CTE's**.
  More info: https://sqlmesh.readthedocs.io/en/latest/concepts/tests/
2. Unlike dbt, SQLMesh does not need any connection to your data warehouse to run these
  tests. By default, it transpiles the SQL syntax of your model into DuckDB SQL and
  **runs your unit tests in-memory using DuckDB**. You can change this in `config.yaml`
  (per model) if this transpilation is unreliable.

Since this demo project is already using DuckDB, the 2nd point is not so useful.

You can run unit tests using `sqlmesh test`

In [ ]:
!cd /workspaces/sqlmesh_playground/ && sqlmesh test

### Exercises

1. Look in `tests/` to see an example of a unit test
2. Modify the test `test_commits`, make it fail. Verify using `sqlmesh test`
3. Run a `sqlmesh plan`, note that it refuses to run if tests fail

## Audits (data tests)

In SQLMesh, data tests are called audits. These are conditions
or business rules that your input and/or output data is expected to satisfy, e.g.
certain columns may not be null, must contain unique values, etc.

Similarly to dbt, you can use built-in tests or define your own. SQLMesh has more tests
built-in than dbt (it has many tests that in dbt are part of packages `dbt_utils` and
`dbt_expectatations`).

The equivalent of this dbt test config:

```yaml
models:
  - name: my_model
    columns:
      - name: id
        tests:
          - not_null
          - unique
```

is

```java
MODEL (
  name my_model,
  audits (
    not_null(columns := (id)),
    unique_values(columns := (id))
  )
);
```

### How audits are run

There is a command `sqlmesh audit` which is similar to `dbt test`, however it does not
take an `--environment` argument, and this is [not planned](https://github.com/TobikoData/sqlmesh/issues/305).

The reason is that unlike dbt, audits are always run as part of a `sqlmesh plan` (or
`sqlmesh run`) when any of these 3 is true:

 - A model's input data changes (new data interval)
 - You change the model's code
 - You change the model's audits (e.g. add one, or change one)

So to run audits if you added a new audit or changed one, simply run `sqlmesh plan` as
before. It won't run all your audits, but you don't want to run them all. You want to
run only the one(s) you changed.

A failing audit will block a plan from being applied. If you want to change that, you
can use a non-blocking version of an audit (e.g. use `not_null_non_blocking` instead
of `not_null`).

Custom audits can be defined in `audits/` and can be used in models. An example can
be found in `audits/assert_nonnegative_num_commits`, used in model `commits_stats_total`.

### Exercises

1. Add a new audit (your choice) to any of the models and "run" the audit (using
  `sqlmesh plan` as before).
2. Add an audit to model `events` that checks that `id` is unique (no duplicates).
  Try to run/plan your code and note that this audit fails. This is expected, there are
  indeed duplicates:

In [ ]:
import notebook_utils
notebook_utils.query_duckdb(
    """
    SELECT *
    FROM persistent.gharchive__dev.events
    WHERE id = 45186813027
    """
)

### Exercises (continued)

3. Change the audit added in exercise 2 to a non-blocking variant. Run it. You should
  get a warning similar to this but the run should proceed:

```log
  [WARNING] gharchive__dev.events: 'unique_values_non_blocking' audit error: 2 rows failed. Learn more in logs: /workspaces/sqlmesh_playground/logs/sqlmesh_2025_12_29_10_53_33.log

  [3/3] gharchive__dev.events    [insert 2025-01-01 02:00:00-02:59:59, audits ✔1 ❌1]                           0.06s
```